# Векторный поиск вакансий

Site config связывает TEI/E5-base и физическую коллекцию Qdrant. Notebook показывает подготовку запроса, текст для модели, аудит tokenizer, тело native query, поиск и CSV batch.

Откройте notebook с Python 3.12+ kernel проекта; Jupyter установите отдельно.
Подготовьте site config по [руководству](../../../docs/usage.md#векторный-поиск). TEI и Qdrant должны быть запущены: notebook не запускает Docker.
Ключи берутся из `TEI_API_KEY` и `QDRANT_API_KEY` окружения kernel. Не помещайте их в ячейки или overrides и не выводите resolved config. Результаты могут содержать документы корпуса; поставляемый notebook сохранён без outputs.


In [ ]:
import json
import sys
from pathlib import Path
from uuid import uuid4

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs/vector.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

from src.core.transformers.data import PipelineData
from src.domain.request.filters import QueryFilter

## Параметры

`site_config` — тот же файл, что использует CLI: URL, pinned image для выбранной GPU, profile/recipe, corpus path и collection. Дополнительные overrides источника и поиска задаются в `site_overrides`.

`build_collection=True` разрешает построить новое поколение с указанным физическим именем. Существующее имя отклоняется; для готовой коллекции оставьте `False`.
Для просмотра текста документа и аудита tokenizer нужен исходный корпус. Выбирайте новый output directory для каждого опыта: JSON/CSV не перезаписываются.


In [ ]:
site_config = Path("/srv/recsys/config/vector-site.yaml")
site_overrides = []
text = "Сварщик без опыта"
input_path = ROOT / "examples/blue-collar/queries.csv"
output_dir = ROOT / "outputs" / "vector-notebook" / uuid4().hex
build_collection = False

## Hydra-композиция

Context manager позволяет повторять ячейку. Каталог site config служит основным, search path добавляет группы checkout. После изменения параметров пересоберите объекты.
Exporters отключены; Compose API не требует CLI runtime Hydra. Пути экспорта задаются ниже.


In [ ]:
def compose_vector(wrapper):
    with initialize_config_dir(
        config_dir=str(site_config.resolve().parent), version_base="1.3"
    ):
        return compose(
            config_name=site_config.stem,
            overrides=[
                f"hydra.searchpath=[file://{ROOT / 'configs'}]",
                f"experiments/vacancy-search/vector@experiment={wrapper}",
                *site_overrides,
                "experiment.exporters=[]",
            ],
        )


cfg = compose_vector("002-interactive")
experiment = instantiate(cfg.experiment, _convert_="all")
leaf = experiment.registry["vector_search"]

## Подготовка запроса без HTTP

`prepare` нормализует query, добавляет `query: ` и строит объявленные scalar-фильтры. План включает context/filters эксперимента, не кодирует текст и не меняет PipelineData.
Дополнительные условия ручного поиска включайте и в план, и в последующий `search`.


In [ ]:
probe = PipelineData(
    context=instantiate(cfg.experiment.context, _convert_="all"),
    filters=[*instantiate(cfg.experiment.filters, _convert_="all"), QueryFilter(text)],
)
plan = leaf.prepare(probe)
print(json.dumps(plan.to_dict(), ensure_ascii=False, indent=2))

## Текст документа и аудит tokenizer

Reader проверяет корпус, renderer выбирает поля по recipe, embedder добавляет `passage: `. Исходные fields сохраняются. Byte limit и token limit проверяются отдельно: запрос длиннее 512 tokens отклоняется, документ может усекаться справа на TEI.

Аудит вызывает `/tokenize` с prefix и special tokens, по одному документу, без truncation. Response budget задаётся Hydra, по умолчанию 16 MiB. Между запросами сохраняются только counts.
Ошибка или превышение бюджета прерывает аудит без отчёта об успехе. Counts показывают объём усечения; для оценки качества нужны поисковые измерения.


In [ ]:
reader = instantiate(cfg.corpus.reader, _convert_="all")
renderer = instantiate(cfg.corpus.renderer, _convert_="all")
documents = reader.read()
example_document = documents[0]
example_text = renderer.render(example_document)
example_model_input = leaf.embedder.prepare_documents((example_text,))[0]
print(
    json.dumps(
        {
            "document_id": example_document.id,
            "model_input": example_model_input,
            "utf8_bytes": len(example_model_input.encode("utf-8")),
        },
        ensure_ascii=False,
        indent=2,
    )
)

In [ ]:
audit_report = None
leaf.embedder.check_runtime()
audit_counts = []
token_preview = []
for document in documents:
    document_text = renderer.render(document)
    model_input = leaf.embedder.prepare_documents((document_text,))[0]
    tokens = leaf.embedder.integration.tokenize((model_input,))[0]
    token_count = len(tokens)
    if not audit_counts:
        token_preview = tokens[:8]
    del tokens
    audit_counts.append({"document_id": document.id, "tokens": token_count})

truncated_count = sum(
    row["tokens"] > leaf.embedder.profile.max_tokens for row in audit_counts
)
audit_report = {
    "documents": len(audit_counts),
    "truncated_documents": truncated_count,
    "truncation_share": truncated_count / len(audit_counts),
    "max_tokens": max(row["tokens"] for row in audit_counts),
    "model_limit": leaf.embedder.profile.max_tokens,
}
print(
    json.dumps({"first_document_tokens": token_preview}, ensure_ascii=False, indent=2)
)
audit_report

## Построение новой коллекции

При `build_collection=True` выполняются preflight, создание indexes, encode/upload batches, read-back и публикация ready. После успеха receipt сохраняется отдельно.
Ошибка до публикации оставляет коллекцию недоступной поиску. Ошибка записи receipt после remote ready не отменяет готовность и не удаляет коллекцию. Повторный build с тем же именем отклоняется.


In [ ]:
if build_collection:
    index_cfg = compose_vector("001-index")
    index_cfg.experiment.exporters = [
        {
            "_target_": "src.components.reporting.exporters.json.JSONExporter",
            "path": str(output_dir / "vector-index.json"),
        }
    ]
    output_dir.mkdir(parents=True, exist_ok=True)
    index = instantiate(index_cfg.experiment, _convert_="all")
    receipt = index.run()
    for exporter in index.exporters:
        exporter.export(receipt)
    print("Ready collection:", index.collection)

## Кодирование запроса и native body

После проверки ready descriptor выполняется один query encode. `plan.body` принимает готовый вектор; полный `native_body` доступен в kernel. Вывод показывает первые координаты и размер.
Следующая ячейка отправляет это тело запроса и возвращает кандидатов до TopK.


In [ ]:
leaf.integration.check_collection(leaf.collection, leaf.source)
query_embedding = leaf.embedder.encode_query(plan.original_query)
native_body = plan.body(query_embedding)
print(
    json.dumps(
        {
            "profile_id": query_embedding.profile_id,
            "dimensions": len(query_embedding.values),
            "vector_head": list(query_embedding.values[:8]),
            "request_settings": {
                key: value for key, value in native_body.items() if key != "query"
            },
        },
        ensure_ascii=False,
        indent=2,
    )
)

In [ ]:
candidates = leaf.integration.query(leaf.collection, native_body, leaf.source)
[
    {"id": item.id, "score": item.score, "fields": item.fields}
    for item in candidates.items
]

## Полный поиск

`SearchExperiment.search` заново кодирует запрос и применяет настроенный TopK. Дополнительные условия передаются через `search(text, context={...}, filters=[...])`; исходный текст остаётся QueryFilter.
Semantic relevance, ANN recall и performance оцениваются отдельными измерениями.


In [ ]:
serp = experiment.search(text)
rows = [
    {"id": item.id, "score": item.score, "fields": item.fields} for item in serp.items
]
rows

## CSV batch и сохранение

Wrapper `003-batch` использует CSV `id,text` и собранный experiment. Весь CSV проверяется до первого запроса; ошибка поиска прекращает batch без частичного экспорта.
`run()` возвращает artifact, CSV сохраняется отдельной операцией. Поля результата: query id/text, rank, document id, score и полный JSON fields.


In [ ]:
batch_cfg = compose_vector("003-batch")
batch_cfg.experiment.input_path = str(input_path)
batch_cfg.experiment.exporters = [
    {
        "_target_": "src.components.reporting.exporters.csv.SearchCSVExporter",
        "path": str(output_dir / "searches.csv"),
    }
]
batch = instantiate(batch_cfg.experiment, search=experiment, _convert_="all")
artifact = batch.run()
artifact.to_rows()[:5]

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
for exporter in batch.exporters:
    exporter.export(artifact)
output_dir / "searches.csv"